# Lending Club P2P Loan Default Prediction — Part 1: Baseline Reproduction

**UE24CS352A Machine Learning — Mini-Project, Team A-15 (Sec-A)**
Aman Priyansh (PES1UG24AM030) and Addanki Harsha Rajeswari (PES1UG24AM014)

Paper reproduced: *LendingClub Loan Default and Profitability Prediction*, Peiqian Li and Gao Han, Stanford CS229, 2018.

## 1. Data and the authors' code

In [ ]:
# CELL 1 - Get the data and the author's code
# 1) Download the Lending Club dataset from Kaggle (about 1.6 GB)
# 2) Download the paper's original code zip from the author's Google Drive
# We only READ the author's code to copy their steps. We do not run it.

import kagglehub, os, glob, zipfile, gdown

data_dir = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Data folder:", data_dir)
for f in glob.glob(data_dir + "/**/*", recursive=True):
    print("  ", f, round(os.path.getsize(f)/1e6, 1), "MB")

gdown.download(id="1EilPoVCYx6fI84FUrfJvxGCUho5TImXx", output="author_code.zip", quiet=True)
with zipfile.ZipFile("author_code.zip") as z:
    z.extractall("author_code")
    print("Author files:", sorted(z.namelist()))

In [ ]:
# CELL 2 - Print the author's key code files so we can copy their exact steps
files = ["utils.py", "features_eng.py", "log_reg.py", "random_forest.py", "classification_eval.py",
         "utils_raw_return_regression.py", "ridge.py", "linear_regression.py", "mlp_regressor.py",
         "random_forest_regressor.py", "random_forest_shallow_regressor_10.py", "regression_eval.py", "regression_cost_eval.py"]
for name in files:
    print("=" * 30, name, "=" * 30)
    print(open("author_code/" + name).read())

## 2. Rebuild the missing cleaned dataset

In [ ]:
# CELL 3 - Rebuild the paper's "data_clean.csv"
# The author's code reads a file called data_clean.csv that is NOT in the zip.
# The paper (Section III-A) says how it was made:
#   - loans issued 2012 to 2015
#   - only loans with a FINAL status: "Fully Paid", "Charged Off", "Default"
#     (so "Current", "Late", "In Grace Period" are removed)
import pandas as pd, numpy as np

csv_file = data_dir + "/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

# All columns the author's classification + regression code uses, plus issue_d to filter by year
author_cols = ['loan_amnt','funded_amnt','total_pymnt','term','int_rate','installment','issue_d','last_pymnt_d',
 'emp_length','home_ownership','annual_inc','verification_status','loan_status','purpose','zip_code','addr_state','dti',
 'delinq_2yrs','earliest_cr_line','fico_range_low','fico_range_high','inq_last_6mths','mths_since_last_delinq',
 'mths_since_last_record','open_acc','pub_rec','revol_bal','revol_util','total_acc','out_prncp','out_prncp_inv',
 'last_fico_range_high','last_fico_range_low','collections_12_mths_ex_med','mths_since_last_major_derog','policy_code',
 'application_type','annual_inc_joint','dti_joint','acc_now_delinq','tot_coll_amt','tot_cur_bal','open_acc_6m',
 'open_act_il','open_il_12m','open_il_24m','mths_since_rcnt_il','total_bal_il','il_util','open_rv_12m','open_rv_24m',
 'all_util','total_rev_hi_lim','inq_fi','total_cu_tl','inq_last_12m','acc_open_past_24mths','avg_cur_bal',
 'bc_open_to_buy','bc_util','chargeoff_within_12_mths','delinq_amnt','mo_sin_old_il_acct','mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','mths_since_recent_bc_dlq',
 'mths_since_recent_inq','mths_since_recent_revol_delinq','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','pct_tl_nvr_dlq','percent_bc_gt_75',
 'pub_rec_bankruptcies','tax_liens','tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit']

raw = pd.read_csv(csv_file, usecols=author_cols, low_memory=False)
print("Full Kaggle file:", raw.shape)

raw["issue_year"] = pd.to_datetime(raw["issue_d"], format="%b-%Y").dt.year
clean = raw[raw["issue_year"].between(2012, 2015) &
            raw["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
clean = clean.drop(columns=["issue_year"])
del raw

print("After filter (paper had 745,529):", len(clean))
print(clean["loan_status"].value_counts())
print((clean["loan_status"] != "Fully Paid").mean().round(3), "= share negative (paper: 0.19)")
clean.to_csv("data_clean.csv", index=False)

## 3. The authors' cleaning steps

In [ ]:
# CELL 4 - The author's cleaning steps (copied from their utils.py)
# Same three fill rules as the paper: mean-fill, zero-fill, and max-fill (max x 5),
# then one-hot encoding, a random 70/30 split, and standardisation fitted on train only.
#
# Changes ONLY so the 2018 code runs today (the maths is identical):
#   - int_rate / revol_util are already numbers in this file (e.g. 13.5), so we just divide by 100
#   - dates: "days since 1970" computed the modern way
#   - MEMORY: the author's pandas code makes several full copies of a 786k x 1,100 table.
#     Here we build ONE float32 table and standardise it in place, so it fits comfortably in RAM.
import pandas as pd, numpy as np, os, gc
if 'clean' not in globals():
    clean = pd.read_csv("data_clean.csv", low_memory=False)
print("Loans:", len(clean))

nan_max_cols = ['mths_since_last_delinq','mths_since_last_record','mths_since_last_major_derog','mths_since_rcnt_il',
 'mths_since_recent_bc','mths_since_recent_bc_dlq','mths_since_recent_inq','mths_since_recent_revol_delinq',
 'mo_sin_old_il_acct','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl']
nan_zero_cols = ['tot_coll_amt','tot_cur_bal','open_acc_6m','open_act_il','open_il_12m','open_il_24m','total_bal_il',
 'open_rv_12m','open_rv_24m','inq_fi','inq_last_12m','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','mort_acc','percent_bc_gt_75',
 'tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit','bc_open_to_buy',
 'acc_open_past_24mths','total_cu_tl','total_rev_hi_lim']
nan_mean_cols = ['dti','annual_inc_joint','dti_joint','il_util','all_util','bc_util','pct_tl_nvr_dlq','avg_cur_bal','revol_util']

def author_clean(df):
    df = df.copy()
    df['revol_util'] = pd.to_numeric(df['revol_util'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    df['int_rate']   = pd.to_numeric(df['int_rate'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    for col in df.columns:
        if col in nan_mean_cols: df[col] = df[col].fillna(np.nanmean(df[col].values))
        if col in nan_zero_cols: df[col] = df[col].fillna(0.0)
        if col in nan_max_cols:  df[col] = df[col].fillna(np.nanmax(df[col].values) * 5)
    d = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
    df['earliest_cr_line'] = (d - pd.Timestamp('1970-01-01')).dt.days
    return df

def build_matrix(df, label_col, seed=0):
    # 1) shuffle rows (author: df.sample(frac=1)); seed fixed so we can re-run and get the same split
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    y = df[label_col].to_numpy(); df = df.drop(columns=[label_col])
    # 2) one-hot encoding (author: pd.get_dummies) - done straight into one float32 table
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = [c for c in df.columns if c not in cat_cols]
    cats = {c: sorted(df[c].dropna().unique()) for c in cat_cols}
    names = num_cols + [c + "_" + str(v) for c in cat_cols for v in cats[c]]
    n, p = len(df), len(names)
    X = np.zeros((n, p), dtype=np.float32)
    X[:, :len(num_cols)] = df[num_cols].to_numpy(dtype=np.float32)
    j = len(num_cols)
    for c in cat_cols:
        codes = pd.Categorical(df[c], categories=cats[c]).codes.astype(np.int64)
        rows = np.where(codes >= 0)[0]
        X[rows, j + codes[rows]] = 1.0
        j += len(cats[c])
    print("Empty (NaN) cells left after the author's fills:", int(np.isnan(X).sum()))
    np.nan_to_num(X, copy=False)
    # 3) 70 / 30 split: first 70% of the shuffled rows = train (author: np.split at 0.7)
    n_tr = int(.7 * n)
    # 4) StandardScaler fitted on TRAIN only, applied to all rows, in place (mean 0, std 1)
    for k in range(0, p, 100):
        blk = X[:n_tr, k:k+100].astype(np.float64)
        m, s = blk.mean(0), blk.std(0); s[s == 0] = 1.0
        X[:, k:k+100] -= m.astype(np.float32); X[:, k:k+100] /= s.astype(np.float32)
    del df; gc.collect()
    return X[:n_tr], y[:n_tr], X[n_tr:], y[n_tr:], names

# ---- Classification data (author's prepare_data) ----
cls_df = author_clean(clean.drop(columns=['total_pymnt','issue_d','last_pymnt_d']))
cls_df['loan_status'] = cls_df['loan_status'].map({'Charged Off': 0, 'Default': 0, 'Fully Paid': 1})
train_x, train_y, test_x, test_y, cls_features = build_matrix(cls_df, 'loan_status')
del cls_df; gc.collect()
print("Train:", train_x.shape, " Test:", test_x.shape)
print("Number of features (paper: 1,097):", len(cls_features))
del clean; gc.collect()   # free 1 GB; data_clean.csv stays on disk for the regression part

## 4. Classification — Logistic Regression and Random Forest

In [ ]:
# CELL 5 - Logistic Regression (paper Section V-A)
# Author's code: LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=1000)
# class_weight='balanced' = defaults (only 19% of loans) count more, so the model does not ignore them
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support

results = {}   # we collect every model's numbers here for the final comparison table

def predict_chunks(model, X, bs=50000):
    # predict 50,000 rows at a time: scikit-learn copies the input to 64-bit inside predict(),
    # so predicting in chunks keeps memory use low
    return np.concatenate([model.predict(X[i:i + bs]) for i in range(0, len(X), bs)])

def report(name, y_true, y_pred, split):
    print("--- " + name + " on " + split + " ---")
    print("Confusion matrix (rows = true Default / Paid Off, columns = predicted Default / Paid Off):")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=["Default", "Paid Off"], digits=2))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred)
    pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    results[(name, split)] = dict(def_p=p[0], def_r=r[0], def_f1=f[0], paid_p=p[1], paid_r=r[1], paid_f1=f[1], w_p=pw, w_r=rw, w_f1=fw)

t = time.time()
log_reg = LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=1000)
log_reg.fit(train_x, train_y)
print("Training took", round((time.time() - t) / 60, 1), "minutes\n")
report("Logistic Regression", train_y, predict_chunks(log_reg, train_x), "train")
report("Logistic Regression", test_y, predict_chunks(log_reg, test_x), "test")

### Neural Network classifier (rebuilt from paper text)

In [ ]:
# CELL 6 - Neural Network classifier (paper Section V-B)
# The zip has NO code file for this model, so we use exactly what the paper text says:
#   4 hidden layers (10, 10, 5, 3), sigmoid activation for all neurons,
#   cross-entropy (log) loss, gradient-descent-based solver (Adam),
#   mini-batch size 200, learning rate 0.001, L2 penalty 0.0001
from sklearn.neural_network import MLPClassifier

t = time.time()
nn_clf = MLPClassifier(hidden_layer_sizes=(10, 10, 5, 3), activation='logistic', solver='adam',
                       batch_size=200, learning_rate_init=0.001, alpha=0.0001, random_state=0)
nn_clf.fit(train_x, train_y)
print("Training took", round((time.time() - t) / 60, 1), "minutes,", nn_clf.n_iter_, "epochs\n")
report("Neural Network", train_y, predict_chunks(nn_clf, train_x), "train")
report("Neural Network", test_y, predict_chunks(nn_clf, test_x), "test")

In [ ]:
# CELL 7 - Random Forest classifier (paper Section V-C)
# Author's code: RandomForestClassifier(n_estimators=200, n_jobs=-1, max_depth=30, max_features=50)
#   200 trees, Gini loss, each split looks at only 50 random features, trees at most 30 levels deep
# The slowest classifier: about 4-5 minutes on an A100 runtime (it uses all CPU cores, n_jobs=-1).
from sklearn.ensemble import RandomForestClassifier

t = time.time()
rf_clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, max_depth=30, max_features=50, random_state=0)
rf_clf.fit(train_x, train_y)
print("Training took", round((time.time() - t) / 60, 1), "minutes\n")
report("Random Forest", train_y, predict_chunks(rf_clf, train_x), "train")
report("Random Forest", test_y, predict_chunks(rf_clf, test_x), "test")

## 5. Regression data preparation and linear models

In [ ]:
# CELL 8 - Regression data: predict the Net Annualized Return (NAR) of each loan (paper Section VI)
# Author's code (utils.py, prepare_regression_data):
#   label = (total_pymnt / funded_amnt) ** (365 / (days from issue_d to last_pymnt_d + 30))
#   This is (1 + NAR). The paper writes NAR = that - 1. Subtracting 1 does not change MSE or R2.
#   Then total_pymnt, issue_d, last_pymnt_d are DROPPED (they were only used to build the label),
#   and loan_status is not used as a feature. Same fills, one-hot, 70/30 split, scaling as before.

# Free the classification table first (each table is ~3.5 GB in memory)
del train_x, test_x, rf_clf; gc.collect()

clean = pd.read_csv("data_clean.csv", low_memory=False)
reg_df = clean.dropna(subset=['last_pymnt_d']).drop(columns=['loan_status']).copy()
days = (pd.to_datetime(reg_df['last_pymnt_d'], format='%b-%Y') - pd.to_datetime(reg_df['issue_d'], format='%b-%Y')).dt.days
reg_df['label'] = (reg_df['total_pymnt'] / reg_df['funded_amnt']) ** (365.0 / (days + 30))
reg_df = reg_df.drop(columns=['total_pymnt', 'issue_d', 'last_pymnt_d'])
reg_df = author_clean(reg_df)
del clean; gc.collect()

rtrain_x, rtrain_y, rtest_x, rtest_y, reg_features = build_matrix(reg_df, 'label')
del reg_df; gc.collect()
print("Loans with a last payment date:", len(rtrain_y) + len(rtest_y))
print("Train:", rtrain_x.shape, " Test:", rtest_x.shape)
print("NAR (label - 1) summary on train:")
print(pd.Series(rtrain_y - 1).describe().round(3))

In [ ]:
# CELL 9 - Linear Regression, Ridge Regression, Neural Network regressor (paper Sections VII-A, B, C)
# Measured with MSE (mean squared error) and R2 (coefficient of determination), on train and test.
#   Linear:  author's code LinearRegression()          (plain least squares, no penalty)
#            copy_X=False only saves RAM: our train table is already centred (mean 0), so nothing changes
#   Ridge:   author's code Ridge()  -> alpha = 1        (least squares + L2 penalty)
#   NN:      paper text: ReLU, Adam, batch 200, L2 penalty 0.0001, hidden layers (20, 10, 5, 3)
#            (note: the author's mlp_regressor.py file uses alpha=0.008 instead - we follow the paper text)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

reg_results = {}
def reg_report(name, model):
    for split, X, y in [("train", rtrain_x, rtrain_y), ("test", rtest_x, rtest_y)]:
        pred = predict_chunks(model, X)
        reg_results[(name, split)] = dict(mse=mean_squared_error(y, pred), r2=r2_score(y, pred))
        print(f"{name:30s} {split:5s}  MSE = {reg_results[(name, split)]['mse']:.4g}   R2 = {reg_results[(name, split)]['r2']:.4g}")

for name, model in [("Linear Regression", LinearRegression(n_jobs=-1, copy_X=False)),
                    ("Ridge Regression (alpha=1)", Ridge(alpha=1.0)),
                    ("Neural Network regressor", MLPRegressor(hidden_layer_sizes=(20, 10, 5, 3), activation='relu',
                                                             solver='adam', batch_size=200, alpha=0.0001, random_state=0))]:
    t = time.time()
    model.fit(rtrain_x, rtrain_y)
    print(f"({name} trained in {(time.time() - t) / 60:.1f} min)")
    reg_report(name, model)
    del model; gc.collect()

## 6. Random Forest regressors on the GPU

In [ ]:
# CELL 10 - Check the machine: GPU, CPUs, RAM
# Why: the Random Forest REGRESSOR looks at all ~1,100 features at every split, on 550k loans.
# That is very slow on CPU, so Cell 11 trains it on the GPU. Use Runtime > Change runtime type > A100 (or L4).
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc
!free -h | head -2

In [ ]:
# CELL 11 - Random Forest regressors at depth 4, 8 and 10 (paper Section VII-D) - on the GPU
# Author's code: RandomForestRegressor(n_estimators=100, max_depth=4 / 8 / 10)  (scikit-learn, CPU)
# On CPU this takes hours per forest, so we train the same kind of forest on the GPU
# with XGBoost's RANDOM-FOREST MODE (XGBRFRegressor). Same idea: 100 independent trees, each on a random sample
# of the loans, predictions averaged. Settings kept: 100 trees, depths 4 / 8 / 10, all features at every split.
# Differences (listed in README "Deviations"):
#   - BINNING: each feature's values are grouped into 128 bins (like grouping exam marks into grade bands) and only
#     bin edges are tried as split points. This is what makes it fast; results move only slightly.
#   - each tree sees a random 63.2% of the loans without repeats (a bootstrap sample has ~63.2% unique loans).
# Why not cuML (NVIDIA's GPU copy of scikit-learn)? We tried it first: in this Colab image its predictions on
# unseen (test) loans were wrong (R2 = -2.26) because of a library version clash, so we used XGBoost instead.
import xgboost as xgb

rf_reg = {}
for depth in [4, 8, 10]:
    t = time.time()
    rf_reg[depth] = xgb.XGBRFRegressor(n_estimators=100, max_depth=depth, subsample=0.632, colsample_bynode=1.0,
                                       max_bin=128, tree_method='hist', device='cuda', random_state=0).fit(rtrain_x, rtrain_y)
    print(f"(depth {depth} trained in {(time.time() - t) / 60:.1f} min on GPU)")
    reg_report(f"Random Forest regressor depth {depth}", rf_reg[depth])

## 7. Investment strategy (paper Section VIII, Figure 2)

In [ ]:
# CELL 12 - Loan selection strategy (paper Section VIII, Figure 2)
# Author's regression_eval.py:
#   1) Use the depth-10 Random Forest to predict every TRAIN loan's return; sort loans from best to worst prediction
#   2) For the top k loans (k = 100, 101, ...), average their TRUE annualized return -> curve "return vs % of loans invested"
#   3) Walk down the curve; the first point where the return falls below 15.6% gives the cut-off prediction M
#   4) On the TEST set, invest only in loans whose prediction >= M, and measure their true average return
# Paper result: M = 0.132 picks about 1.7% of loans and earns about 15% a year (train and test).
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

model = rf_reg[10]

def return_curve(X, y):
    pred = predict_chunks(model, X)
    order = np.argsort(-pred)
    nar_sorted = y[order] - 1.0                       # true NAR of loans, best prediction first
    k = np.arange(1, len(y) + 1)
    avg_return = np.cumsum(nar_sorted) / k            # average NAR of the top-k loans
    return pred, order, avg_return[99:], k[99:] / len(y)   # author starts the curve at k = 100

train_pred, train_order, train_curve, train_frac = return_curve(rtrain_x, rtrain_y)
test_pred,  test_order,  test_curve,  test_frac  = return_curve(rtest_x,  rtest_y)

# Step 3: cut-off from the TRAIN curve (author's code: first point below 0.156)
i = int(np.argmax(train_curve < 0.156))
cutoff = np.sort(train_pred)[::-1][i + 99]
print(f"Cut-off prediction from train curve: M = {cutoff - 1:.3f}   (paper: M = 0.132)")

# Step 4: apply the cut-off to both sets
strategy = {}
for split, pred, y in [("train", train_pred, rtrain_y), ("test", test_pred, rtest_y)]:
    picked = pred >= cutoff
    strategy[split] = dict(share=picked.mean(), ret=(y[picked] - 1).mean(), n=int(picked.sum()))
    print(f"{split:5s}: invest in {picked.sum():6d} loans = {picked.mean()*100:.2f}% of loans, "
          f"true annualized return = {(y[picked] - 1).mean()*100:.1f}%   (paper: ~1.7% of loans, 15%)")

# Paper's own threshold M = 0.132, applied directly
for split, pred, y in [("train", train_pred, rtrain_y), ("test", test_pred, rtest_y)]:
    picked = pred >= 1.132
    print(f"With the paper's M = 0.132 on {split}: {picked.mean()*100:.2f}% of loans, return = {(y[picked] - 1).mean()*100:.1f}%")

# Figure 2 recreated: train and test on ONE chart, different colours
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_frac * 100, train_curve * 100, color="#2a78d6", linewidth=2, label="Train set")
ax.plot(test_frac * 100, test_curve * 100, color="#eb6834", linewidth=2, label="Test set")
ax.axvline(strategy["test"]["share"] * 100, color="#888888", linestyle="--", linewidth=1)
ax.text(strategy["test"]["share"] * 100 + 1, ax.get_ylim()[1] * 0.9,
        f"cut-off: {strategy['test']['share']*100:.1f}% of loans", color="#444444")
ax.set_title("Annualized return vs. percentage of loans invested (Random Forest regressor, depth 10)")
ax.set_xlabel("Percentage of loans invested (best predictions first)")
ax.set_ylabel("Average true annualized return (NAR)")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig("figure2_return_vs_invested.png", dpi=150)
plt.show()